# RVC Voice Model Training — c-elo Kikuyu Voice

Trains an RVC (Retrieval-based Voice Conversion) model from your **celo-sliced** WAV chunks.
The result is a `.pth` model + `.index` file you drop into `rvc-server/` on your machine.

**Before you start:**
- Runtime → Change runtime type → **T4 GPU** (free) or A100 (Colab Pro)
- Your chunks are 40 kHz mono — we use the **40k pretrain** weights throughout

### What each step does
| Step | What happens |
|---|---|
| 1 | Mount Google Drive |
| 2 | Check GPU + verify chunks |
| 3 | Clone RVC repo + install deps |
| 4 | Download 40k pretrained weights |
| 5 | Copy your chunks into RVC dataset folder |
| 6 | Extract features (pitch + HuBERT embeddings) |
| 7 | Write filelist |
| 8 | Train the voice model |
| 9 | Build FAISS index |
| 10 | Listen — quick inference test |
| 11 | Save model to Drive + download |
| 12 | Use on your machine |


## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CHUNKS = '/content/drive/MyDrive/rvc-celo/dataset'
wavs = [f for f in os.listdir(DRIVE_CHUNKS) if f.endswith('.wav')]
print(f'Found {len(wavs)} WAV files in Drive')
assert len(wavs) > 10, 'Too few WAVs — check the upload path above'

## Step 2 — Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU — Runtime → Change runtime type → T4 GPU')

## Step 3 — Clone RVC and install dependencies

**Strategy:** install all deps explicitly, including fairseq from GitHub source
(PyPI fairseq fails to build on pip ≥ 24). Takes ~5–8 minutes on first run.

In [ ]:
import os, sys

RVC_DIR  = '/content/RVC'
EXP_NAME = 'celo'
SR       = '40k'

# 1. Clone official RVC repo
if not os.path.exists(RVC_DIR):
    !git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git {RVC_DIR}
else:
    print('RVC already cloned')
%cd {RVC_DIR}

# 2. Install runtime deps
!pip install -q librosa soundfile praat-parselmouth ffmpeg-python torchcrepe
!pip install -q av
import importlib
try:
    importlib.import_module('av')
    print('av (PyAV) installed OK')
except ImportError:
    print('av not found — retrying with pinned version')
    !pip install -q 'av==10.0.0'
    importlib.invalidate_caches()
    importlib.import_module('av')
    print('av==10.0.0 installed OK')
!pip install -q pyworld==0.3.4
!pip install -q faiss-gpu
!pip install -q tensorboard tqdm scipy scikit-learn

# 3. Install fairseq (if not already present) then patch for Python 3.12 + Colab.
#    a) dataclass/configs.py: mutable defaults forbidden in Python 3.12
#    b) logging/progress_bar.py: top-level tensorboard import triggers
#       tensorflow->jax->numpy ABI crash on Colab
import os, glob, sys, site

def _find_fairseq():
    for sp in site.getsitepackages() + ['/usr/local/lib/python3.12/dist-packages']:
        p = os.path.join(sp, 'fairseq')
        if os.path.isdir(p):
            return p
    return None

FAIRSEQ_DIR = _find_fairseq()
if FAIRSEQ_DIR is None:
    print('fairseq not found — installing from GitHub source...')
    !pip install -q 'omegaconf>=2.1,<2.2' hydra-core bitarray sacrebleu sacremoses sentencepiece
    !pip install -q --no-deps 'git+https://github.com/facebookresearch/fairseq.git@v0.12.2'
    FAIRSEQ_DIR = _find_fairseq()
    assert FAIRSEQ_DIR, 'fairseq install failed — check network and retry'
    print(f'fairseq installed at: {FAIRSEQ_DIR}')
else:
    print(f'fairseq found at: {FAIRSEQ_DIR}')
    !pip install -q 'omegaconf>=2.1,<2.2' hydra-core bitarray sacrebleu sacremoses sentencepiece

# patch a: configs.py — wrap mutable defaults in field(default_factory=...)
CONFIGS_PATH = f'{FAIRSEQ_DIR}/dataclass/configs.py'
with open(CONFIGS_PATH) as f:
    src = f.read()
if 'from dataclasses import dataclass' in src and ', field' not in src:
    src = src.replace('from dataclasses import dataclass', 'from dataclasses import dataclass, field')
elif 'from dataclasses import' in src and 'field' not in src:
    src = src.replace('from dataclasses import', 'from dataclasses import field,', 1)
PATCHES_A = [
    ('common: CommonConfig = CommonConfig()',               'common: CommonConfig = field(default_factory=CommonConfig)'),
    ('common_eval: CommonEvalConfig = CommonEvalConfig()',  'common_eval: CommonEvalConfig = field(default_factory=CommonEvalConfig)'),
    ('distributed_training: DistributedTrainingConfig = DistributedTrainingConfig()', 'distributed_training: DistributedTrainingConfig = field(default_factory=DistributedTrainingConfig)'),
    ('dataset: DatasetConfig = DatasetConfig()',            'dataset: DatasetConfig = field(default_factory=DatasetConfig)'),
    ('optimization: OptimizationConfig = OptimizationConfig()', 'optimization: OptimizationConfig = field(default_factory=OptimizationConfig)'),
    ('checkpoint: CheckpointConfig = CheckpointConfig()',  'checkpoint: CheckpointConfig = field(default_factory=CheckpointConfig)'),
    ('bmuf: FairseqBMUFConfig = FairseqBMUFConfig()',      'bmuf: FairseqBMUFConfig = field(default_factory=FairseqBMUFConfig)'),
    ('generation: GenerationConfig = GenerationConfig()',  'generation: GenerationConfig = field(default_factory=GenerationConfig)'),
    ('eval_lm: EvalLMConfig = EvalLMConfig()',             'eval_lm: EvalLMConfig = field(default_factory=EvalLMConfig)'),
    ('interactive: InteractiveConfig = InteractiveConfig()', 'interactive: InteractiveConfig = field(default_factory=InteractiveConfig)'),
    ('ema: EMAConfig = EMAConfig()',                       'ema: EMAConfig = field(default_factory=EMAConfig)'),
]
na = sum(1 for old, _ in PATCHES_A if old in src)
for old, new in PATCHES_A:
    src = src.replace(old, new)
with open(CONFIGS_PATH, 'w') as f:
    f.write(src)
for pyc in glob.glob(f'{FAIRSEQ_DIR}/dataclass/__pycache__/configs*.pyc'):
    os.remove(pyc)
print(f'patch a: configs.py — {na}/11 fields patched')

# patch b: progress_bar.py — guard tensorboard import inside a lazy loader
PB_PATH = f'{FAIRSEQ_DIR}/logging/progress_bar.py'
with open(PB_PATH) as f:
    pb = f.read()
OLD_TB = '''try:
    _tensorboard_writers = {}
    from torch.utils.tensorboard import SummaryWriter
except ImportError:
    try:
        from tensorboardX import SummaryWriter
    except ImportError:
        SummaryWriter = None'''
NEW_TB = '''_tensorboard_writers = {}
SummaryWriter = None  # loaded lazily to avoid tensorflow/jax import at startup

def _load_summary_writer():
    global SummaryWriter
    if SummaryWriter is not None:
        return
    try:
        from torch.utils.tensorboard import SummaryWriter as _SW
        SummaryWriter = _SW
    except Exception:
        try:
            from tensorboardX import SummaryWriter as _SW
            SummaryWriter = _SW
        except Exception:
            pass'''
nb_count = 1 if OLD_TB in pb else 0
pb = pb.replace(OLD_TB, NEW_TB)
OLD_WRITER = '''    def _writer(self, key):
        if SummaryWriter is None:
            return None'''
NEW_WRITER = '''    def _writer(self, key):
        _load_summary_writer()
        if SummaryWriter is None:
            return None'''
if OLD_WRITER in pb:
    pb = pb.replace(OLD_WRITER, NEW_WRITER)
    nb_count += 1
with open(PB_PATH, 'w') as f:
    f.write(pb)
for pyc in glob.glob(f'{FAIRSEQ_DIR}/logging/__pycache__/progress_bar*.pyc'):
    os.remove(pyc)
print(f'patch b: progress_bar.py — {nb_count}/2 sections patched')

# Clear cached fairseq modules from any previous import attempt
for key in list(sys.modules.keys()):
    if key.startswith('fairseq'):
        del sys.modules[key]

import importlib
importlib.invalidate_caches()
import fairseq, fairseq.checkpoint_utils
print(f'fairseq OK: {fairseq.__version__}')

# 4. Verify all deps
ok = True
for label, mod in [
    ('torch',          'torch'),
    ('fairseq',        'fairseq'),
    ('faiss',          'faiss'),
    ('librosa',        'librosa'),
    ('soundfile',      'soundfile'),
    ('av',             'av'),
    ('parselmouth',    'parselmouth'),
    ('pyworld',        'pyworld'),
]:
    try:
        importlib.import_module(mod)
        print(f'  \u2713 {label}')
    except ImportError as e:
        print(f'  \u2717 {label} — {e}')
        ok = False
print('\n\u2705 Ready — continue to Step 4' if ok else '\n\u26a0\ufe0f  Fix \u2717 items above')


## Step 4 — Download 40k pretrained weights

Uses the **40k** pretrained G/D weights because your chunks are 40 kHz mono.
Also downloads `hubert_base.pt` and `rmvpe.pt` for feature extraction.

In [ ]:
import os

BASE_URL = 'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main'

files_needed = [
    'pretrained_v2/f0G40k.pth',
    'pretrained_v2/f0D40k.pth',
    'hubert_base.pt',
    'rmvpe.pt',
]

for rel_path in files_needed:
    dest = f'{RVC_DIR}/{rel_path}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        print(f'Downloading {rel_path}...')
        !wget -q --show-progress -O {dest} {BASE_URL}/{rel_path}
    else:
        print(f'Already have {rel_path}')

# Verify all files exist and are non-empty
for rel_path in files_needed:
    dest = f'{RVC_DIR}/{rel_path}'
    size = os.path.getsize(dest) if os.path.exists(dest) else 0
    status = '✓' if size > 100_000 else '✗ MISSING/EMPTY'
    print(f'  {status}  {rel_path}  ({size/1e6:.1f} MB)')

print('\nAll 40k pretrain weights ready')

## Step 5 — Copy chunks into RVC dataset folder

In [ ]:
import os, shutil
from tqdm.notebook import tqdm

DATASET_DEST = f'{RVC_DIR}/dataset/{EXP_NAME}'
os.makedirs(DATASET_DEST, exist_ok=True)

wavs = sorted([f for f in os.listdir(DRIVE_CHUNKS) if f.endswith('.wav')])
print(f'Copying {len(wavs)} WAVs from Drive → {DATASET_DEST}')

for fname in tqdm(wavs):
    src  = os.path.join(DRIVE_CHUNKS, fname)
    dest = os.path.join(DATASET_DEST, fname)
    if not os.path.exists(dest):
        shutil.copy2(src, dest)

copied = len([f for f in os.listdir(DATASET_DEST) if f.endswith('.wav')])
print(f'\nDataset ready: {copied} WAVs in {DATASET_DEST}')
assert copied > 10, 'Too few WAVs copied — check Drive path'

## Step 6 — Extract features

Three sub-steps: loudness normalisation → HuBERT content embeddings → RMVPE pitch (F0).
Takes ~10–20 min for 395 chunks.

In [ ]:
%cd {RVC_DIR}

# Guard: ensure av (PyAV) is importable in THIS kernel before launching preprocess.
# pip installs from Step 3 don't always propagate to subprocesses on first run;
# re-running here is fast (no-op if already cached) and guarantees the wheel is present.
import subprocess, sys, importlib
def _ensure_av():
    try:
        import av
        print(f'av already available: {av.__version__}')
        return
    except ImportError:
        pass
    print('av not found — installing now...')
    ret = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'av'],
        capture_output=True, text=True
    )
    importlib.invalidate_caches()
    try:
        import av
        print(f'av installed OK: {av.__version__}')
        return
    except ImportError:
        pass
    # Latest wheel unavailable for this Python — try pinned release
    print('Retrying with av==10.0.0...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'av==10.0.0'],
        capture_output=True, text=True
    )
    importlib.invalidate_caches()
    import av  # raises ImportError clearly if still missing
    print(f'av==10.0.0 installed OK: {av.__version__}')

_ensure_av()

# Create the log dir preprocess.py expects before it runs
import os
os.makedirs(f'{RVC_DIR}/logs/{EXP_NAME}', exist_ok=True)

# 6a: preprocess — normalise loudness, resample to 16k mono
!python infer/modules/train/preprocess.py \
    {DATASET_DEST} 40000 2 {RVC_DIR}/logs/{EXP_NAME} False 3.0

import glob
wav16 = glob.glob(f'{RVC_DIR}/logs/{EXP_NAME}/1_16k_wavs/*.wav')
print(f'\nPreprocessed: {len(wav16)} files in 1_16k_wavs/')
assert len(wav16) > 0, 'Preprocessing produced 0 files — check av (PyAV) install and ffmpeg on PATH'

In [ ]:
%cd {RVC_DIR}

# 6b: HuBERT content feature extraction
# Discover the actual script path — repo layout varies by branch/commit
import glob as _glob, os as _os
def _find_script(rvc_dir, *candidates):
    for name in candidates:
        hits = _glob.glob(f'{rvc_dir}/**/{name}', recursive=True)
        if hits:
            return hits[0]
    return None

feat_script = _find_script(
    RVC_DIR,
    'extract_feature_print.py',  # most branches
    'extract_feature_input_print.py',  # some forks
    'extract_features.py',
)
print(f'Feature script: {feat_script}')
assert feat_script, f'Could not find HuBERT feature extraction script in {RVC_DIR} — check the repo cloned in Step 3'

# Correct call signature (from script source):
#   7 args: device n_part i_part exp_dir version is_half
#   8 args: device n_part i_part i_gpu exp_dir version is_half
# We use the 8-arg form to set CUDA_VISIBLE_DEVICES explicitly.
#
# Script loads HuBERT from assets/hubert/hubert_base.pt relative to cwd.
# Ensure that path exists before running.
import os as _os
_os.makedirs(f'{RVC_DIR}/assets/hubert', exist_ok=True)
_hubert_src = f'{RVC_DIR}/hubert_base.pt'
_hubert_dst = f'{RVC_DIR}/assets/hubert/hubert_base.pt'
if _os.path.exists(_hubert_src) and not _os.path.exists(_hubert_dst):
    import shutil as _shutil
    _shutil.copy2(_hubert_src, _hubert_dst)
    print(f'Copied hubert_base.pt → assets/hubert/')
assert _os.path.exists(_hubert_dst), f'hubert_base.pt not found at {_hubert_dst} — re-run Step 4'

LOG_DIR = f'{RVC_DIR}/logs/{EXP_NAME}'
!python {feat_script} cuda:0 1 0 0 {LOG_DIR} v2 False

import glob
feats = glob.glob(f'{RVC_DIR}/logs/{EXP_NAME}/3_feature256/*.npy')
print(f'\nHuBERT features: {len(feats)} .npy files')

In [ ]:
%cd {RVC_DIR}

# 6c: RMVPE F0 pitch extraction
# Discover the actual script path — repo layout varies by branch/commit
f0_script = _find_script(
    RVC_DIR,
    'extract_f0_rmvpe.py',
    'extract_f0.py',
)
print(f'F0 script: {f0_script}')
assert f0_script, f'Could not find F0 extraction script in {RVC_DIR} — check the repo cloned in Step 3'

!python {f0_script} 1 0 0 {RVC_DIR}/logs/{EXP_NAME} True

import glob
f0s = glob.glob(f'{RVC_DIR}/logs/{EXP_NAME}/2a_f0/*.npy')
print(f'\nF0 files: {len(f0s)} .npy files')

## Step 7 — Write filelist

In [ ]:
import os, glob

LOG_DIR = f'{RVC_DIR}/logs/{EXP_NAME}'
feature_files = sorted(glob.glob(f'{LOG_DIR}/1_16k_wavs/*.wav'))
print(f'Found {len(feature_files)} preprocessed audio files')
assert len(feature_files) > 0, 'No files — Step 6a must complete successfully first'

filelist_path = f'{LOG_DIR}/filelist.txt'
with open(filelist_path, 'w') as f:
    for fp in feature_files:
        f.write(fp + '\n')
print(f'Written: {filelist_path}')

## Step 8 — Train the RVC model

| Setting | Value | Why |
|---|---|---|
| `total_epoch` | 200 | Good for ~395 chunks; try 300 if you have time |
| `batch_size` | 8 | Safe for T4 16 GB; use 12 on A100 |
| `sr` | 40k | Matches 40 kHz chunks |
| `f0method` | rmvpe | Best pitch accuracy |
| `version` | v2 | Better architecture |

~200 epochs on T4 takes **45–90 minutes**.

In [ ]:
TOTAL_EPOCH = 200
BATCH_SIZE  = 8
SAVE_EVERY  = 10

import subprocess, sys
cmd = [
    sys.executable,
    f'{RVC_DIR}/infer/modules/train/train.py',
    '-e',  EXP_NAME,
    '-sr', SR,
    '-f0', '1',
    '-bs', str(BATCH_SIZE),
    '-g',  '0',
    '-te', str(TOTAL_EPOCH),
    '-se', str(SAVE_EVERY),
    '-pg', f'{RVC_DIR}/pretrained_v2/f0G40k.pth',
    '-pd', f'{RVC_DIR}/pretrained_v2/f0D40k.pth',
    '-l',  '0',
    '-c',  '0',
    '-sw', '0',
    '-v',  'v2',
]
print('Starting training...  (this will take a while)')
result = subprocess.run(cmd, cwd=RVC_DIR)
print('\nTraining exit code:', result.returncode)

## Step 9 — Build FAISS retrieval index

In [ ]:
%cd {RVC_DIR}

!python infer/modules/train/extract_index.py \
    {RVC_DIR}/logs/{EXP_NAME} v2

import glob
index_files = glob.glob(f'{RVC_DIR}/logs/{EXP_NAME}/*.index')
print('Index files created:')
for f in index_files:
    print(f'  {f}')

## Step 10 — Quick inference test

Synthesize a Kikuyu phrase with base MMS-TTS, then pass it through your trained RVC model.

In [ ]:
import glob, os

pth_files = sorted(glob.glob(f'{RVC_DIR}/weights/{EXP_NAME}.pth'))
if not pth_files:
    pth_files = sorted(glob.glob(f'{RVC_DIR}/logs/{EXP_NAME}/G_*.pth'))
latest_pth = pth_files[-1] if pth_files else None
index_files = glob.glob(f'{RVC_DIR}/logs/{EXP_NAME}/*.index')
index_file  = index_files[0] if index_files else ''

print(f'Model : {latest_pth}')
print(f'Index : {index_file}')

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch, soundfile as sf
from IPython.display import Audio, display

device = 'cuda'
tok = AutoTokenizer.from_pretrained('facebook/mms-tts-kik')
mms = VitsModel.from_pretrained('facebook/mms-tts-kik').to(device).eval()

test_text = 'Ndi mwega, ndi na gikeno'
inputs    = tok(test_text, return_tensors='pt').to(device)
with torch.no_grad():
    mms_wav = mms(**inputs).waveform[0].cpu().numpy()

TEST_INPUT = '/content/test_input.wav'
sf.write(TEST_INPUT, mms_wav, mms.config.sampling_rate)
print('MMS-TTS output (before RVC):')
display(Audio(mms_wav, rate=mms.config.sampling_rate))

In [ ]:
TEST_OUTPUT = '/content/test_output_celo.wav'

!python {RVC_DIR}/infer/infer.py \
    --input_path   {TEST_INPUT} \
    --opt_path     {TEST_OUTPUT} \
    --model_path   {latest_pth} \
    --index_path   {index_file} \
    --f0method     rmvpe \
    --f0up_key     0 \
    --index_rate   0.75 \
    --filter_radius 3 \
    --resample_sr  0 \
    --rms_mix_rate 0.25 \
    --protect      0.33

data, sr = sf.read(TEST_OUTPUT)
print('After RVC (c-elo voice):')
display(Audio(data, rate=sr))

## Step 11 — Save model to Drive + download

In [ ]:
import shutil, os, glob

DRIVE_OUT = '/content/drive/MyDrive/rvc-celo/output'
os.makedirs(DRIVE_OUT, exist_ok=True)

weight_candidates = (
    glob.glob(f'{RVC_DIR}/weights/{EXP_NAME}.pth') +
    sorted(glob.glob(f'{RVC_DIR}/logs/{EXP_NAME}/G_*.pth'))
)

dest_pth = dest_idx = None
if weight_candidates:
    final_pth = weight_candidates[-1]
    dest_pth  = f'{DRIVE_OUT}/celo.pth'
    shutil.copy2(final_pth, dest_pth)
    print(f'Saved model : {dest_pth}')
else:
    print('No .pth found — training may not have completed')

idx_files = glob.glob(f'{RVC_DIR}/logs/{EXP_NAME}/*.index')
if idx_files:
    dest_idx = f'{DRIVE_OUT}/{os.path.basename(idx_files[0])}'
    shutil.copy2(idx_files[0], dest_idx)
    print(f'Saved index : {dest_idx}')

print('\nFiles at: My Drive/rvc-celo/output/')

In [ ]:
# Optional: download directly to your browser
from google.colab import files
if dest_pth: files.download(dest_pth)
if dest_idx: files.download(dest_idx)

## Step 12 — Use on your machine

```
rvc-server/
  models/
    celo.pth
    celo.index     ← rename from added_IVF*.index
```

Update `rvc-server/main.py`:
```python
MODEL_PATH = 'models/celo.pth'
INDEX_PATH = 'models/celo.index'
```

Restart the server — **Engine R** on the Speak page will now use the c-elo voice.

### Inference parameter tuning
| Param | Default | Effect |
|---|---|---|
| `index_rate` | 0.75 | Lower if voice sounds grainy |
| `f0up_key` | 0 | ±1–2 semitones if pitch sounds off |
| `protect` | 0.33 | Higher if consonants sound distorted |
| `filter_radius` | 3 | Higher if pitch wobbles |
